<a id="top"></a>

# Demo: cost is per token, both directions, every call

```yaml
title: "Demo: cost is per token, both directions, every call"
keywords: cost, pricing, input output asymmetry, conversation history, order of magnitude, teacher demo
estimated duration: 15
```

> **Lesson:** L01. Teacher demo — Demo 4 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L01/demos_or_activities.md). Runs offline. ⚠️ Rates below are illustrative — confirm current pricing before teaching.

## Contents

- [1. Setup](#1-setup)
- [2. The input/output asymmetry](#2-the-inputoutput-asymmetry)
- [3. The conversation-history staircase](#3-the-conversation-history-staircase)
- [4. Order-of-magnitude ladder](#4-order-of-magnitude-ladder)

## 1. Setup

In [1]:
# Illustrative rates only. ALWAYS confirm current Claude Sonnet pricing on Anthropic's
# pricing page on the day you teach — a stale slide undermines every cost claim that follows.
INPUT_USD_PER_MTOK = 3.00
OUTPUT_USD_PER_MTOK = 15.00  # output is the EXPENSIVE direction (~3-5x input)


def call_cost_usd(input_tokens: int, output_tokens: int) -> float:
    """Cost of one call in USD, given token counts and per-MILLION-token rates."""
    return (
        input_tokens / 1_000_000 * INPUT_USD_PER_MTOK
        + output_tokens / 1_000_000 * OUTPUT_USD_PER_MTOK
    )


print("setup ready")

setup ready


[↑ Back to top](#top)

## 2. The input/output asymmetry

Output tokens cost more than input. A long prompt + short answer is often cheaper than a short prompt + long answer.

In [2]:
print("long input,  short answer:", f"${call_cost_usd(2000, 50):.5f}")
print("short input, long answer: ", f"${call_cost_usd(50, 2000):.5f}  <- output is pricier")

long input,  short answer: $0.00675
short input, long answer:  $0.03015  <- output is pricier


[↑ Back to top](#top)

## 3. The conversation-history staircase

Each turn re-sends everything before it, so input tokens climb even when each message is small.

In [3]:
PER_TURN = 200  # new tokens added per turn
cumulative = 0
total_cost = 0.0
for t in range(1, 6):
    cumulative += PER_TURN
    cost = call_cost_usd(cumulative, 60)
    total_cost += cost
    print(f"turn {t}: input re-sent ~{cumulative:4} tokens -> ${cost:.5f}")
print(f"session total: ${total_cost:.5f}  (history is re-billed every turn)")

turn 1: input re-sent ~ 200 tokens -> $0.00150
turn 2: input re-sent ~ 400 tokens -> $0.00210
turn 3: input re-sent ~ 600 tokens -> $0.00270
turn 4: input re-sent ~ 800 tokens -> $0.00330
turn 5: input re-sent ~1000 tokens -> $0.00390
session total: $0.01350  (history is re-billed every turn)


[↑ Back to top](#top)

## 4. Order-of-magnitude ladder

The jump from 'free' to 'a real budget' is the punchline — do this multiplication *before* running agents.

In [4]:
one_call = call_cost_usd(2000, 300)
print(f"1 call                       ~ ${one_call:.5f}")
print(f"x10  (a 10-step agent run)   ~ ${one_call * 10:.4f}")
print(f"x100 (dev iteration)         ~ ${one_call * 10 * 100:.2f}")
print(f"x1000 (small deployment)     ~ ${one_call * 10 * 100 * 1000:,.2f}")

1 call                       ~ $0.01050
x10  (a 10-step agent run)   ~ $0.1050
x100 (dev iteration)         ~ $10.50
x1000 (small deployment)     ~ $10,500.00


[↑ Back to top](#top)